# Collaborative Tree Search (CoTS) | Advanced Planning & Search

In [1]:
from langchain_openai import ChatOpenAI
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import List, Optional
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
import json
import re

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
@dataclass
class TreeNode:
    label: str
    content: str = ""
    score: float = 0.0
    explored_by: str = ""
    children: List["TreeNode"] = field(default_factory=list)

    def best_child(self) -> Optional["TreeNode"]:
        return max(self.children, key=lambda c: c.score) if self.children else None

    def pretty(self, indent: int = 0) -> str:
        tag = f"[{self.explored_by} | score={self.score}]" if self.explored_by else ""
        lines = [f"{'  ' * indent}{self.label} {tag}"]
        for c in self.children:
            lines.append(c.pretty(indent + 1))
        return "\n".join(lines)

BRANCHES = {
    "Agent-A": "push notifications (mobile/web)",
    "Agent-B": "email notifications",
    "Agent-C": "webhook notifications",
}

# --- Phase 1: parallel initial exploration (each agent owns one branch) ---
def explore_branch(agent: str, branch: str, problem: str) -> TreeNode:
    resp = model.invoke(
        f"You are {agent}, an expert in {branch}.\nProblem: {problem}\n\n"
        "Propose a design for the {branch} subsystem. Cover: architecture, key components, "
        "trade-offs, and scalability.\nThen rate your own design 0-10 for quality. "
        'Reply as JSON: {{"design": "...", "score": <number>}}'
    )
    try:
        data = json.loads(re.search(r"\{.*\}", resp.content, re.S).group())
    except Exception:
        data = {}
    node = TreeNode(label=branch, content=data.get("design", resp.content),
                    score=float(data.get("score", 5)), explored_by=agent)
    return node

def run_collaborative_tree_search(problem: str) -> str:
    root = TreeNode(label="Notification System Design")

    # Phase 1 — parallel exploration
    print("Phase 1: parallel branch exploration...")
    with ThreadPoolExecutor(max_workers=3) as pool:
        futures = {a: pool.submit(explore_branch, a, b, problem) for a, b in BRANCHES.items()}
        for agent, fut in futures.items():
            root.children.append(fut.result())
    for c in root.children:
        print(f"  {c.label}: score={c.score} by {c.explored_by}")

    # Phase 2 — cross-examination (each agent reviews others' branches and may adjust scores)
    print("\nPhase 2: cross-examination...")
    summaries = "\n".join(
        f"- {c.label} (score {c.score}): {c.content[:300]}" for c in root.children
    )
    for child in root.children:
        resp = model.invoke(
            f"You are reviewing designs for: {problem}\n\n"
            f"All branch summaries:\n{summaries}\n\n"
            f"Re-score the '{child.label}' branch (0-10) considering how it compares to "
            "the others. Reply with ONLY a number."
        )
        try:
            child.score = float(re.search(r"[\d.]+", resp.content).group())
        except Exception:
            pass
    print("  Updated scores:", {c.label: c.score for c in root.children})

    # Phase 3 — deep-dive: ALL agents collaborate on the winning branch
    winner = root.best_child()
    print(f"\nPhase 3: deep-dive on winner '{winner.label}' (score={winner.score})...")
    deep_resp = model.invoke(
        f"The winning branch for '{problem}' is: {winner.label}\n"
        f"Current design:\n{winner.content}\n\n"
        "Three senior architects now collaborate to deepen this design.\n"
        "Add: detailed component diagram, failure handling, scaling strategy, "
        "and integration points with the other notification channels "
        f"({', '.join(c.label for c in root.children if c is not winner)}).\n"
        "Be specific and actionable."
    )
    deep_node = TreeNode(label=f"{winner.label} (deep-dive)", content=deep_resp.content,
                         score=winner.score, explored_by="All-Agents")
    winner.children.append(deep_node)

    # Phase 4 — final synthesis picking best elements from every branch
    print("\nPhase 4: final synthesis...")
    all_content = "\n\n".join(
        f"### {c.label} (score {c.score}):\n{c.content[:500]}" for c in root.children
    )
    synthesis = model.invoke(
        f"Problem: {problem}\n\n{all_content}\n\n"
        f"Deep-dive on winner:\n{deep_node.content[:800]}\n\n"
        "Which elements from EACH branch should be preserved in the final design? "
        "Identify the strongest contribution from each agent's exploration. "
        "Then synthesize them into one cohesive design, incorporating the winning "
        "branch as the core while preserving each branch's unique strengths."
    )
    print("\n" + root.pretty())
    return synthesis.content

In [5]:
# --- Run ---
result = run_collaborative_tree_search("Design a notification system for 1M users with minimal latency")
print(f"\n{'='*60}\nFinal synthesized design:\n{result[:800]}")

Phase 1: parallel branch exploration...
  push notifications (mobile/web): score=8.0 by Agent-A
  email notifications: score=8.0 by Agent-B
  webhook notifications: score=8.0 by Agent-C

Phase 2: cross-examination...
  Updated scores: {'push notifications (mobile/web)': 8.5, 'email notifications': 7.0, 'webhook notifications': 8.0}

Phase 3: deep-dive on winner 'push notifications (mobile/web)' (score=8.5)...

Phase 4: final synthesis...

Notification System Design 
  push notifications (mobile/web) [Agent-A | score=8.5]
    push notifications (mobile/web) (deep-dive) [All-Agents | score=8.5]
  email notifications [Agent-B | score=7.0]
  webhook notifications [Agent-C | score=8.0]

Final synthesized design:
To design a cohesive notification system for 1 million users with minimal latency, we will integrate the strongest contributions from each branch: push notifications, email notifications, and webhook notifications. The core of the design will be based on the push notifications branc